In [1]:
"""
TMDB Trending Movies Scraper
Mengambil data dari endpoint Trending TMDB (max 1000 halaman = 20.000 film)
"""

import requests
import pandas as pd
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
# ── CONFIG ────────────────────────────────────────────────────────────────
API_KEY  = "a7eaa982f861eda59593b7ebf834464c"
PAGES    = 500
BASE_URL = "https://api.themoviedb.org/3"

In [3]:
# Setup retry session
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504]
)
session.mount("https://", HTTPAdapter(max_retries=retry_strategy))

In [4]:
def get_now_playing_ids(page):
    """Fetch satu halaman dari endpoint /movie/now_playing."""
    r = session.get(
        f"{BASE_URL}/movie/now_playing",
        params={"api_key": API_KEY, "language": "en-US", "page": page},
        timeout=15,
    )
    r.raise_for_status()
    return r.json().get("results", [])

In [5]:
def get_movie_details(movie_id):
    """Fetch detail lengkap + credits untuk satu film."""
    r = session.get(
        f"{BASE_URL}/movie/{movie_id}",
        params={
            "api_key": API_KEY,
            "language": "en-US",
            "append_to_response": "credits,release_dates",
        },
        timeout=15,
    )
    r.raise_for_status()
    return r.json()

In [6]:
def parse_certificate(release_dates):
    """Ambil rating usia US (e.g. PG-13)."""
    for country in release_dates.get("results", []):
        if country.get("iso_3166_1") == "US":
            for rd in country.get("release_dates", []):
                cert = rd.get("certification", "").strip()
                if cert:
                    return cert
    return None

In [7]:
def parse_movie(data):
    """Ekstrak semua field dari response detail film."""
    title    = data.get("title")
    year     = (data.get("release_date") or "")[:4] or None
    overview = data.get("overview") or None
 
    certificate = parse_certificate(data.get("release_dates", {}))
 
    runtime  = data.get("runtime")
    duration = f"{runtime} min" if runtime else None
 
    genres = data.get("genres", [])
    genre  = ", ".join(g["name"] for g in genres)
 
    tmdb_rating = data.get("vote_average")
    votes       = data.get("vote_count")
 
    credits  = data.get("credits", {})
    crew     = credits.get("crew", [])
    cast     = credits.get("cast", [])
 
    director = next((p["name"] for p in crew if p.get("job") == "Director"), None)
    stars    = [p["name"] for p in cast[:4]]
 
    revenue = data.get("revenue")
    grossed = f"${revenue:,}" if revenue else None
 
    return {
        "Movie Name":        title,
        "Year":              year,
        "Certificate":       certificate,
        "Duration":          duration,
        "Genre":             genre,
        "TMDB Rating":       tmdb_rating,
        "Votes":             votes,
        "Director":          director,
        "Stars":             stars,
        "Grossed in $":      grossed,
        "Plot":              overview,
        "Original Language": data.get("original_language"),
        "Popularity":        data.get("popularity"),
        "TMDB ID":           data.get("id"),
        "IMDb ID":           data.get("imdb_id"),
    }

In [8]:
# ── Main ──────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # Step 1: kumpulkan semua movie ID
    all_ids = []
    for page in range(1, PAGES + 1):
        try:
            results = get_now_playing_ids(page)
            if not results:
                print(f"  Halaman {page}: tidak ada hasil, berhenti.")
                break
            all_ids.extend(m["id"] for m in results)
            if page % 100 == 0:
                print(f"  Halaman {page}/{PAGES} — {len(all_ids)} ID terkumpul")
        except Exception as e:
            print(f"  Halaman {page} gagal: {e}")
        time.sleep(0.25)
 
    all_ids = list(dict.fromkeys(all_ids))
    print(f"\nTotal unique ID: {len(all_ids)}")
    print("Fetching detail tiap film...\n")
 
    # Step 2: fetch detail tiap film dengan checkpoint setiap 500 film
    checkpoint_path = "../data/raw/tmdb_now_playing_checkpoint.parquet"
    movie_data = []
 
    for idx, movie_id in enumerate(all_ids):
        try:
            data    = get_movie_details(movie_id)
            details = parse_movie(data)
            movie_data.append(details)
 
            if (idx + 1) % 500 == 0:
                pd.DataFrame(movie_data).to_parquet(checkpoint_path, index=False)
                print(f"[{idx+1}/{len(all_ids)}] ✓ checkpoint disimpan — {details['Movie Name']}")
        except Exception as e:
            print(f"[{idx+1}/{len(all_ids)}] ✗ ID {movie_id}: {e}")
 
        time.sleep(0.26)
 
    # Step 3: simpan final
    df = pd.DataFrame(movie_data)
    df.to_parquet("../data/raw/tmdb_movies_now_playing_raw.parquet", index=False)
    print(f"\nSelesai! {len(df)} film disimpan ke tmdb_movies_now_playing_raw.parquet")

  Halaman 100/500 — 2000 ID terkumpul
  Halaman 200/500 — 4000 ID terkumpul
  Halaman 240: tidak ada hasil, berhenti.

Total unique ID: 4658
Fetching detail tiap film...

[500/4658] ✓ checkpoint disimpan — Kaneko Fumiko
[1000/4658] ✓ checkpoint disimpan — Bluey at the Cinema: Playdates with Friends
[1500/4658] ✓ checkpoint disimpan — Love Letter to Texas
[2000/4658] ✓ checkpoint disimpan — When Birds Can't Fly
[2500/4658] ✓ checkpoint disimpan — All of Her
[3000/4658] ✓ checkpoint disimpan — Hands Up Who'd Like a Hug?
[3500/4658] ✓ checkpoint disimpan — Life Is a Beautiful Game
[3935/4658] ✗ ID 1659953: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/movie/1659953?api_key=a7eaa982f861eda59593b7ebf834464c&language=en-US&append_to_response=credits%2Crelease_dates (Caused by ResponseError('too many 502 error responses'))
[4000/4658] ✓ checkpoint disimpan — Cruda Verdad Dura Moral
[4500/4658] ✓ checkpoint disimpan — Dragon Year Spring Festival

S